In [9]:
import sqlite3
import pandas as pd

In [15]:
import pandas as pd
from sqlalchemy import create_engine

In [16]:
engine = create_engine("sqlite:///../data/bluestock_mf.db")

In [17]:
with open("../sql/schema.sql", "r") as f:
    schema = f.read()

with engine.begin() as conn:
    for statement in schema.split(";"):
        if statement.strip():
            conn.exec_driver_sql(statement)

In [18]:
fund_df = pd.read_csv("../data/raw/01_fund_master.csv")

fund_df.to_sql(
    "dim_fund",
    engine,
    if_exists="append",
    index=False
)

print("Fund Master loaded successfully!")

Fund Master loaded successfully!


In [10]:
conn = sqlite3.connect("../data/mutual_funds.db")

print("Database created successfully!")

Database created successfully!


In [11]:
with open("../sql/schema.sql", "r") as file:
    schema = file.read()

conn.executescript(schema)

print("Tables created successfully!")

Tables created successfully!


In [12]:
tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

tables

,name
0,dim_fund
1,fact_nav
2,fact_transactions
3,fact_performance


In [19]:
nav_df = pd.read_csv("../data/cleaned/clean_nav.csv")

nav_df = nav_df.rename(columns={"date": "nav_date"})

nav_df.to_sql(
    "fact_nav",
    engine,
    if_exists="append",
    index=False
)

print("NAV History loaded successfully!")

NAV History loaded successfully!


In [13]:
fund_df = pd.read_csv("../data/raw/01_fund_master.csv")

fund_df.to_sql(
    "dim_fund",
    conn,
    if_exists="append",
    index=False
)

print("Fund Master loaded successfully!")

Fund Master loaded successfully!


In [21]:
performance_df = pd.read_csv("../data/cleaned/clean_performance.csv")

performance_df.to_sql(
    "fact_performance",
    engine,
    if_exists="append",
    index=False
)

print("Performance dataset loaded successfully!")

Performance dataset loaded successfully!


In [20]:
transactions_df = pd.read_csv("../data/cleaned/clean_transactions.csv")

transactions_df.to_sql(
    "fact_transactions",
    engine,
    if_exists="append",
    index=False
)

print("Transactions loaded successfully!")

Transactions loaded successfully!


In [22]:
from sqlalchemy import text

tables = [
    "dim_fund",
    "fact_nav",
    "fact_transactions",
    "fact_performance"
]

with engine.connect() as conn:
    for table in tables:
        result = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        )
        print(f"{table}: {result.scalar()} rows")

dim_fund: 40 rows
fact_nav: 46000 rows
fact_transactions: 32778 rows
fact_performance: 40 rows


In [23]:
from sqlalchemy import text

query = """
SELECT
    scheme_name,
    fund_house,
    aum_crore
FROM fact_performance
ORDER BY aum_crore DESC
LIMIT 5;
"""

pd.read_sql(text(query), engine)

,scheme_name,fund_house,aum_crore
0,Mirae Asset Emerging Bluechip Fund - Regular -...,Mirae Asset MF,49046.0
1,Kotak Emerging Equity Fund - Regular - Growth,Kotak Mahindra MF,47469.0
2,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,43630.0
3,DSP Top 100 Equity Fund - Regular - Growth,DSP Mutual Fund,41828.0
4,UTI Mid Cap Fund - Regular - Growth,UTI Mutual Fund,41728.0


In [14]:
engine.dispose()

In [24]:
query = """
SELECT
    strftime('%Y-%m', nav_date) AS month,
    ROUND(AVG(nav), 2) AS average_nav
FROM fact_nav
GROUP BY strftime('%Y-%m', nav_date)
ORDER BY month;
"""

pd.read_sql(text(query), engine)

,month,average_nav
0,2022-01,207.06
1,2022-02,207.72
2,2022-03,209.69
3,2022-04,211.83
4,2022-05,212.73
5,2022-06,213.86
6,2022-07,213.96
7,2022-08,215.68
8,2022-09,218.49
9,2022-10,219.53


In [14]:
nav_df = pd.read_csv("../data/cleaned/clean_nav.csv")

nav_df = nav_df.rename(columns={"date": "nav_date"})

nav_df.to_sql(
    "fact_nav",
    conn,
    if_exists="append",
    index=False
)

print("NAV History loaded successfully!")

NAV History loaded successfully!


In [25]:
query = """
SELECT
    strftime('%Y', transaction_date) AS year,
    ROUND(SUM(amount_inr), 2) AS sip_inflow
FROM fact_transactions
WHERE transaction_type = 'Sip'
GROUP BY strftime('%Y', transaction_date)
ORDER BY year;
"""

pd.read_sql(text(query), engine)

,year,sip_inflow
0,2024,153233052.0
1,2025,64000439.0


In [26]:
query = """
SELECT
    state,
    COUNT(*) AS total_transactions
FROM fact_transactions
GROUP BY state
ORDER BY total_transactions DESC;
"""

pd.read_sql(text(query), engine)

,state,total_transactions
0,Punjab,2965
1,Madhya Pradesh,2931
2,Tamil Nadu,2806
3,Gujarat,2780
4,West Bengal,2748
5,Haryana,2736
6,Telangana,2718
7,Uttar Pradesh,2695
8,Delhi,2677
9,Karnataka,2621


In [27]:
query = """
SELECT
    scheme_name,
    fund_house,
    expense_ratio_pct
FROM fact_performance
WHERE expense_ratio_pct < 1
ORDER BY expense_ratio_pct;
"""

pd.read_sql(text(query), engine)

,scheme_name,fund_house,expense_ratio_pct
0,Nippon India Gilt Securities Fund - Regular - ...,Nippon India MF,0.55
1,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,0.56
2,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra MF,0.60
3,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,0.66
4,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,0.72
5,Nippon India Large Cap Fund - Direct - Growth,Nippon India MF,0.72
6,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,0.74
7,Axis Bluechip Fund - Direct - Growth,Axis Mutual Fund,0.75
8,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,0.77
9,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,0.78


In [15]:
transactions_df = pd.read_csv("../data/cleaned/clean_transactions.csv")

transactions_df.to_sql(
    "fact_transactions",
    conn,
    if_exists="append",
    index=False
)

print("Transactions loaded successfully!")

Transactions loaded successfully!


In [28]:
from sqlalchemy import text

query = """
SELECT
    fund_house,
    COUNT(*) AS total_schemes
FROM dim_fund
GROUP BY fund_house
ORDER BY total_schemes DESC;
"""

pd.read_sql(text(query), engine)

,fund_house,total_schemes
0,SBI Mutual Fund,5
1,Nippon India MF,5
2,ICICI Prudential MF,5
3,HDFC Mutual Fund,5
4,Kotak Mahindra MF,4
5,Axis Mutual Fund,4
6,UTI Mutual Fund,3
7,Mirae Asset MF,3
8,DSP Mutual Fund,3
9,Aditya Birla Sun Life MF,3


In [16]:
performance_df = pd.read_csv("../data/cleaned/clean_performance.csv")

performance_df.to_sql(
    "fact_performance",
    conn,
    if_exists="append",
    index=False
)

print("Performance data loaded successfully!")

Performance data loaded successfully!


In [29]:
query = """
SELECT
    category,
    COUNT(*) AS total_schemes
FROM dim_fund
GROUP BY category
ORDER BY total_schemes DESC;
"""

pd.read_sql(text(query), engine)

,category,total_schemes
0,Equity,34
1,Debt,6


In [17]:
tables = [
    "dim_fund",
    "fact_nav",
    "fact_transactions",
    "fact_performance"
]

for table in tables:
    count = pd.read_sql(
        f"SELECT COUNT(*) AS total FROM {table}",
        conn
    )
    print(f"\n{table}")
    print(count)


dim_fund
   total
0     40

fact_nav
   total
0  46000

fact_transactions
   total
0  32778

fact_performance
   total
0     40


In [30]:
query = """
SELECT
    risk_category,
    COUNT(*) AS total_schemes
FROM dim_fund
GROUP BY risk_category
ORDER BY total_schemes DESC;
"""

pd.read_sql(text(query), engine)

,risk_category,total_schemes
0,Moderate,16
1,High,8
2,Very High,6
3,Low,6
4,Moderately High,4


In [31]:
query = """
SELECT
    payment_mode,
    COUNT(*) AS total_transactions
FROM fact_transactions
GROUP BY payment_mode
ORDER BY total_transactions DESC;
"""

pd.read_sql(text(query), engine)

,payment_mode,total_transactions
0,Net Banking,8250
1,Cheque,8228
2,UPI,8154
3,Mandate,8146


In [32]:
query = """
SELECT
    kyc_status,
    COUNT(*) AS total_investors
FROM fact_transactions
GROUP BY kyc_status
ORDER BY total_investors DESC;
"""

pd.read_sql(text(query), engine)

,kyc_status,total_investors
0,Verified,30146
1,Pending,2632
